# OpenFlight Development: Launch Monitor, Force Plate & 3D MoCap Analytics

Welcome to the **OpenFlight Development Technical Notebook**. This notebook integrates multi-sensor physics: high-speed optical vision & Doppler radar processing, tri-axial Ground Reaction Force (GRF) plate dynamics, and 3D optical motion capture kinematic chain calculations.

---

## 1. Sensor Physics & Data Architecture

```
                       +-----------------------------------+
                       |    Athlete & Club Impact Event    |
                       +-----------------+-----------------+
                                         |
            +----------------------------+----------------------------+
            |                            |                            |
            v                            v                            v
 +----------------------+    +-----------------------+    +-----------------------+
 |  High-Speed Vision   |    |    Doppler Radar      |    |   Force Plate Array   |
 |  Stereo Pair         |    |    24/60 GHz FMCW     |    |   Tri-Axial Load Cells|
 |  Impact Vector & Spin|    |    Velocity & Range   |    |   GRF Vector & COP    |
 +----------+-----------+    +-----------+-----------+    +-----------+-----------+
            |                            |                            |
            +----------------------------+----------------------------+
                                         |
                                         v
                       +-----------------------------------+
                       | Synchronized Telemetry Pipeline   |
                       +-----------------------------------+
```

---

## 2. Force Plate Dynamics & Center of Pressure (COP)

Tri-axial force sensors record vertical force $F_z$, lateral shear $F_x$, and anterior-posterior shear $F_y$.

Center of Pressure (COP) coordinates on the plate surface are:

$$
x_{COP} = \frac{-M_y + F_x z_0}{F_z}, \quad y_{COP} = \frac{M_x + F_y z_0}{F_z}
$$

where $M_x, M_y$ are measured moments and $z_0$ is the plate surface height offset.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Generate synthetic Ground Reaction Force (GRF) and COP data during swing (1000 Hz)
t_swing = np.linspace(0, 1.2, 1200)  # 1.2 seconds downswing to followthrough
body_weight = 80.0 * 9.81  # ~785 N

# Vertical force Fz (Lead vs Trail foot)
Fz_lead = body_weight * (0.4 + 0.9 * np.exp(-(((t_swing - 0.55) / 0.15) ** 2)))
Fz_trail = body_weight * (0.6 - 0.4 * (t_swing > 0.4) * np.exp(-(((t_swing - 0.6) / 0.2) ** 2)))

# Center of Pressure Trajectory (x_cop, y_cop) in meters
cop_x = -0.15 + 0.35 * (1.0 / (1 + np.exp(-12 * (t_swing - 0.5))))
cop_y = -0.05 + 0.10 * np.sin(np.pi * t_swing / 1.2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.8))

# Subplot 1: GRF Vertical Forces
ax1.plot(t_swing * 1000, Fz_lead, label="Lead Foot $F_z$ (N)", color="#2980b9", lw=2.5)
ax1.plot(t_swing * 1000, Fz_trail, label="Trail Foot $F_z$ (N)", color="#c0392b", lw=2.5)
ax1.axvline(x=550, color="#7f8c8d", linestyle="--", label="Impact (t0)")
ax1.set_title("Ground Reaction Force (GRF) Profiles", fontsize=12, fontweight="bold")
ax1.set_xlabel("Time (ms)", fontsize=11)
ax1.set_ylabel("Vertical Force (N)", fontsize=11)
ax1.grid(True, linestyle="--", alpha=0.6)
ax1.legend(frameon=True)

# Subplot 2: COP Trajectory Path
sc = ax2.scatter(cop_x * 100, cop_y * 100, c=t_swing * 1000, cmap="viridis", s=15)
cbar = plt.colorbar(sc, ax=ax2)
cbar.set_label("Time (ms)")
ax2.set_title("Combined Center of Pressure (COP) Trace", fontsize=12, fontweight="bold")
ax2.set_xlabel("Lateral Shift X (cm)", fontsize=11)
ax2.set_ylabel("Anterior-Posterior Y (cm)", fontsize=11)
ax2.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

## 3. Launch Monitor Metrics & MoCap Kinematics

Sensor metrics extracted at impact $t_0$:
- **Club Speed ($v_c$)**: Clubhead origin velocity vector at impact.
- **Ball Speed ($v_b$)**: Initial ball launch velocity magnitude.
- **Smash Factor**: $\eta = \frac{v_b}{v_c}$.
- **Spin Axis**: Inclination angle of spin vector $\boldsymbol{\omega}$ relative to horizontal plane.


In [ ]:
# Summary telemetry data table computation
telemetry_data = {
    "Metric": [
        "Club Speed (mph)",
        "Ball Speed (mph)",
        "Smash Factor",
        "Launch Angle (deg)",
        "Spin Rate (rpm)",
        "Spin Axis (deg)",
    ],
    "Measured Value": [112.4, 166.8, 1.484, 10.8, 2350, -2.1],
    "Target Benchmark": [114.0, 168.0, 1.480, 11.0, 2400, 0.0],
}

print("=== OPENFLIGHT MULTI-SENSOR SYNTHESIS SUMMARY ===")
for m, val, tgt in zip(
    telemetry_data["Metric"], telemetry_data["Measured Value"], telemetry_data["Target Benchmark"]
):
    print(f"{m:<25}: Measured = {val:>7.1f} | Benchmark = {tgt:>7.1f}")